# MediciMess_Charlie
## Phase 1 — Data Analysis and Validation

### Analyst
Leigh

### Objective
The purpose of Phase 1 is to understand and validate the Medici Bank transaction dataset before developing the data pipeline.

This analysis will focus on:

- Understanding the structure of the transaction data
- Reviewing data types
- Identifying missing or duplicate data
- Validating double-entry accounting
- Identifying potential data quality issues

## 1. Dataset Preparation

The MediciMess repository initially provided transaction data in both CSV and JSON formats.

Before beginning the analysis, the number of transaction records in each file was validated.

### Initial Dataset

- CSV: 20,000 transaction records
- JSON: 20,000 transaction records

The project includes `generate_additional_data.py`, which appends approximately 60,000 additional historically themed transactions to the existing dataset and updates both the CSV and JSON files.

The expanded dataset will be used for the Phase 1 analysis.

### Expanded Dataset Validation

After running `generate_additional_data.py`, the dataset size was verified in both formats.

- CSV file: 80,231 total lines
- CSV transaction records: 80,230
- JSON transaction records: 80,230

The CSV contains one header row, so the actual number of transaction records is 80,230.

The matching record counts confirm that the expanded CSV and JSON datasets contain the same number of transactions.

In [1]:
import pandas as pd

df = pd.read_csv("../medici_transactions.csv")

In [2]:
df.shape

(80230, 13)

## 2. Dataset Structure

### Dataset Size

The CSV file was loaded into a pandas DataFrame to verify that the complete dataset could be read successfully.

The DataFrame contains:

- **80,230 rows**
- **13 columns**

The 80,230 DataFrame rows match the transaction counts previously verified in both the CSV and JSON files. This confirms that pandas successfully loaded the complete expanded dataset.

### Column Structure

The next step is to identify the columns contained in the dataset and understand what information each field represents.

In [3]:
df.columns

Index(['branch', 'counterparty', 'credit_account', 'credit_account_2',
       'credit_amount', 'credit_amount_2', 'currency', 'date', 'debit_account',
       'debit_amount', 'description', 'id', 'type'],
      dtype='str')

### Columns Identified

The dataset contains 13 fields describing each banking transaction:

| Column | Description |
|---|---|
| `id` | Unique identifier for the transaction |
| `date` | Date the transaction occurred |
| `branch` | Medici Bank branch associated with the transaction |
| `type` | Type/category of transaction |
| `counterparty` | Person, organization, or entity involved in the transaction |
| `description` | Description of the transaction |
| `debit_account` | Account receiving the debit entry |
| `debit_amount` | Amount recorded as the debit |
| `credit_account` | Primary account receiving the credit entry |
| `credit_amount` | Amount recorded as the primary credit |
| `credit_account_2` | Optional secondary account receiving a credit entry |
| `credit_amount_2` | Optional secondary credit amount |
| `currency` | Currency used for the transaction |

### Sample Transactions

A sample of the dataset was reviewed to understand how the fields are populated and how individual transactions are represented.

In [4]:
df.head()

,branch,counterparty,credit_account,credit_account_2,credit_amount,credit_amount_2,currency,date,debit_account,debit_amount,description,id,type
0,Florence,Republic of Florence,Cash,NaN,82833.66,NaN,florin,1390-01-01,Loans Receivable - Government,82833.66,Emergency war financing for Florence defense,1,war_financing
1,Florence,Republic of Florence,Cash,NaN,124432.65,NaN,florin,1390-01-01,Loans Receivable - Government,124432.65,Loan to Venice for Lombardy Wars operations,2,war_financing
2,Bruges,Gold Merchant,Cash,NaN,42534.83,NaN,florin,1390-01-01,Loans Receivable,42534.83,Loan issued to Gold Merchant from Bruges branch,3,loan_issuance
3,Florence,Republic of Florence,Cash,NaN,1617678.46,NaN,florin,1390-01-01,Loans Receivable - Government,1617678.46,War financing for Florentine operations agains...,4,war_financing
4,Florence,Republic of Florence,Cash,NaN,27742.84,NaN,florin,1390-01-01,Loans Receivable - Government,27742.84,Loan to Venice for Lombardy Wars operations,5,war_financing


### Initial Observations

Reviewing the first five transaction records provides an initial view of how the dataset is structured.

- Each row represents an individual banking transaction.
- Each transaction contains an ID, date, branch, counterparty, transaction type, description, debit information, credit information, and currency.
- The sample includes both `war_financing` and `loan_issuance` transaction types.
- The sample transactions use `florin` as the currency.
- `credit_account_2` and `credit_amount_2` contain null (`NaN`) values in the sample. These fields appear to be optional when a transaction does not require a secondary credit entry.
- In the sample records, the debit amount and primary credit amount are equal when no secondary credit is present.

These observations are based only on the initial five records and will be validated against the complete dataset during further analysis.

## 3. Data Types

### Question

What data type did pandas assign to each of the 13 fields when the CSV was loaded?

In [5]:
df.dtypes

branch                  str
counterparty            str
credit_account          str
credit_account_2        str
credit_amount       float64
credit_amount_2     float64
currency                str
date                    str
debit_account           str
debit_amount        float64
description             str
id                    int64
type                    str
dtype: object

### Findings

The dataset contains a combination of text, numeric, and integer fields.

- Transaction IDs were loaded as integers (`int64`).
- Debit and credit amounts were loaded as numeric decimal values (`float64`).
- Descriptive and categorical fields such as branch, counterparty, transaction type, and account names were loaded as strings.
- The `date` field was loaded as a string rather than a datetime value.

The date field may require transformation before performing time-based analysis.

## 4. Missing Value Analysis

### Question

Which columns contain missing values, and do those missing values represent data-quality problems or expected optional fields?

In [6]:
df.isnull().sum()

branch                  0
counterparty            0
credit_account          0
credit_account_2    65632
credit_amount           0
credit_amount_2     65632
currency                0
date                    0
debit_account           0
debit_amount            0
description             0
id                      0
type                    0
dtype: int64

### Findings

Missing values were found only in the optional secondary credit fields:

- `credit_account_2`: 65,632 missing values
- `credit_amount_2`: 65,632 missing values

All other columns contain zero missing values.

The matching null counts for `credit_account_2` and `credit_amount_2` are consistent with the transaction structure. When a transaction does not require a secondary credit entry, both the secondary credit account and secondary credit amount are left blank.

Of the 80,230 transactions:

- 65,632 do not contain a secondary credit entry.
- 14,598 contain a secondary credit entry.

Based on this initial analysis, the missing values appear to occur in expected optional fields rather than required transaction fields.

### Secondary Credit Consistency

Because the two secondary credit fields contain the same number of missing values, we will verify that they are missing or populated together for each transaction.

In [7]:
secondary_credit_mismatch = (
    df["credit_account_2"].isnull()
    !=
    df["credit_amount_2"].isnull()
)

secondary_credit_mismatch.sum()

np.int64(0)

### Secondary Credit Consistency Finding

The secondary credit fields were compared row-by-row to verify that the account and amount fields are either both populated or both missing.

**Mismatched transactions found: 0**

This confirms that `credit_account_2` and `credit_amount_2` are consistently paired throughout the dataset.

The 65,632 missing values previously identified in each field therefore represent expected optional secondary credit entries rather than inconsistent missing data.

## 5. Duplicate Analysis

### Question

Does each transaction have a unique transaction ID, or are any transaction IDs duplicated in the dataset?

Transaction IDs should uniquely identify individual transaction records.

In [8]:
df["id"].duplicated().sum()

np.int64(0)

### Duplicate ID Finding

**Duplicate transaction IDs found: 0**

All 80,230 transaction records have unique transaction IDs. This confirms that the `id` field uniquely identifies each transaction in the dataset.

### Exact Duplicate Records

In addition to checking transaction IDs, the complete dataset will be checked for records where every field is duplicated.

In [9]:
df.duplicated().sum()

np.int64(0)

### Exact Duplicate Record Finding

**Exact duplicate records found: 0**

No records were found where every field was duplicated.

The dataset currently shows no exact duplicate records and no duplicate transaction identifiers.

## 6. Double-Entry Accounting Validation

### Accounting Rule

Each transaction in the MediciMess dataset should follow the double-entry accounting rule:

**Debit Amount = Primary Credit Amount + Secondary Credit Amount**

The `credit_amount_2` field is optional. When no secondary credit exists, its null value will be treated as zero for validation purposes.

### Question

Do all 80,230 transactions satisfy the required double-entry accounting equation?

In [10]:
secondary_credit = df["credit_amount_2"].fillna(0)

In [11]:
secondary_credit.head()

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: credit_amount_2, dtype: float64

In [12]:
total_credit = df["credit_amount"] + secondary_credit

In [13]:
total_credit.head()

0      82833.66
1     124432.65
2      42534.83
3    1617678.46
4      27742.84
dtype: float64

In [14]:
is_balanced = df["debit_amount"] == total_credit

In [15]:
is_balanced.value_counts()

True     76662
False     3568
Name: count, dtype: int64

In [16]:
balance_difference = df["debit_amount"] - total_credit

In [17]:
balance_difference[~is_balanced].head(10)

10     1.818989e-12
21    -3.637979e-12
78    -2.910383e-11
85    -7.275958e-12
100    5.820766e-11
125    9.094947e-13
145    4.547474e-13
146   -4.547474e-13
151    7.275958e-12
157   -3.637979e-12
dtype: float64

### Floating-Point Precision

An initial exact comparison identified 3,568 transactions where the debit amount did not exactly equal the calculated total credit.

Further examination showed that the differences were extremely small and occurred beyond the decimal precision used by the transaction amounts. These differences are consistent with floating-point representation rather than actual accounting imbalances.

Because the financial amounts in the dataset are recorded to two decimal places, the accounting validation will compare debit and credit totals at two-decimal precision.

In [18]:
is_balanced_rounded = (
    df["debit_amount"].round(2)
    ==
    total_credit.round(2)
)

is_balanced_rounded.value_counts()

True     80228
False        2
Name: count, dtype: int64

In [19]:
df.loc[
    ~is_balanced_rounded,
    [
        "id",
        "date",
        "branch",
        "type",
        "debit_account",
        "debit_amount",
        "credit_account",
        "credit_amount",
        "credit_account_2",
        "credit_amount_2"
    ]
]

,id,date,branch,type,debit_account,debit_amount,credit_account,credit_amount,credit_account_2,credit_amount_2
49792,49793,1421-09-02,Milan,loan_repayment,Cash,2231.845,Loans Receivable,2028.95,Interest Income,202.895
78505,78506,1439-11-30,London,loan_repayment,Cash,44973.725,Loans Receivable,35978.98,Interest Income,8994.745


### Precision Consideration

The initial exact comparison produced differences caused by floating-point representation.

A second validation using two-decimal rounding still identified two transactions as unequal. Inspection showed that both transactions were mathematically balanced but contained amounts recorded to three decimal places.

Because the dataset contains values with more than two decimal places, rounding to two decimals before comparison can introduce artificial mismatches.

A tolerance-based comparison will therefore be used to determine whether debit and credit totals are materially equal.

In [20]:
import numpy as np

is_balanced_final = np.isclose(
    df["debit_amount"],
    total_credit,
    atol=1e-9,
    rtol=0
)

pd.Series(is_balanced_final).value_counts()

True    80230
Name: count, dtype: int64

### Double-Entry Accounting Validation Findings

All **80,230 transactions** passed the double-entry accounting validation when evaluated using a tolerance appropriate for floating-point numeric values.

During validation, an exact comparison initially identified 3,568 apparent mismatches. Investigation showed that these differences were caused by floating-point representation and were extremely small.

A two-decimal rounding comparison reduced the apparent mismatches to two transactions. Further inspection confirmed that both transactions were mathematically balanced but contained amounts recorded to three decimal places.

A final tolerance-based comparison using `numpy.isclose()` confirmed:

- **Total transactions evaluated:** 80,230
- **Balanced transactions:** 80,230
- **Unbalanced transactions:** 0

### Conclusion

The dataset passes double-entry accounting validation. For every transaction, the debit amount equals the combined primary and secondary credit amounts within an appropriate numeric tolerance.

The analysis also identified that some transaction amounts contain three decimal places. This should be considered when designing financial validation logic in later phases of the MediciMess pipeline.